In [0]:
from pyspark.sql.functions import current_timestamp
def add_ingestion_date(input_df):
    output_df = input_df.withColumn("ingestion_date", current_timestamp())
    return output_df

In [0]:
#def rearrange_partition_column(input_df, partition_column):
#    cols = [c for c in input_df.columns if c != partition_column]
#    cols.append(partition_column)
#    return input_df.select(*cols) 

In [0]:
from delta.tables import DeltaTable

def merge_delta_data(
    input_df,
    db_name,
    table_name,
    folder_path,
    merge_condition,
    partition_column
):
    spark.conf.set("spark.databricks.optimizer.dynamicPartitionPruning", "true")

    full_table_name = f"{db_name}.{table_name}"

    if spark.catalog.tableExists(full_table_name):
        deltatable = DeltaTable.forName(spark, full_table_name)

        deltatable.alias("tgt") \
            .merge(
                input_df.alias("src"),
                merge_condition
            ) \
            .whenMatchedUpdateAll() \
            .whenNotMatchedInsertAll() \
            .execute()
    else:
        input_df.write \
            .mode("overwrite") \
            .partitionBy(partition_column) \
            .format("delta") \
            .saveAsTable(full_table_name)